# Tutorial 01 — Quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eddmpython/vectrix/blob/master/notebooks/tutorials/01_quickstart.ipynb)

**Build your first forecast in under a minute.** No configuration, no boilerplate, no model selection — just pass your data and get predictions with confidence intervals.

Vectrix evaluates 30+ statistical models (ETS, ARIMA, Theta, CES, DOT, and more), validates each on a holdout set, and returns the best one — all in a single function call.

| What you'll learn | Time |
|:------------------|:-----|
| Forecast from a list | 1 min |
| Forecast from a DataFrame | 1 min |
| Compare all models | 1 min |
| Export results | 1 min |

In [ ]:
!pip install -q vectrix

## 1. Forecast from a List

The simplest possible forecast. Pass a Python list and the number of future steps.

In [ ]:
from vectrix import forecast

data = [120, 135, 148, 130, 155, 170, 162, 180, 195, 185, 200, 215]
result = forecast(data, steps=5)

print(f"Best model: {result.model}")
print(f"Predictions: {result.predictions}")
print(f"MAPE: {result.mape:.2f}%")

What happened behind the scenes:
1. Vectrix inferred frequency from the data length
2. Split data into training (80%) and validation (20%)
3. Fitted 30+ models on the training portion
4. Ranked models by validation MAPE and selected the winner
5. Re-fitted the winner on the full dataset and generated predictions with 95% confidence intervals

## 2. Forecast from a DataFrame

Real-world data usually comes as a DataFrame with date and value columns. Vectrix auto-detects both, or you can specify them.

In [ ]:
from vectrix import forecast, loadSample

df = loadSample("airline")
print(f"Data: {len(df)} rows")
df.tail()

In [ ]:
result = forecast(df, date="date", value="passengers", steps=12)
print(result.summary())

## 3. Supported Input Formats

Vectrix accepts five input formats. No manual conversion needed.

In [ ]:
import numpy as np
import pandas as pd
from vectrix import forecast

values = [100, 120, 130, 115, 140, 160, 150, 170, 190, 180, 200, 215]

r1 = forecast(values, steps=3)                          # Python list
r2 = forecast(np.array(values), steps=3)                # NumPy array
r3 = forecast(pd.Series(values), steps=3)               # Pandas Series

print(f"List     -> {r1.model}: {r1.predictions.round(1)}")
print(f"ndarray  -> {r2.model}: {r2.predictions.round(1)}")
print(f"Series   -> {r3.model}: {r3.predictions.round(1)}")

## 4. Full Text Summary

The `.summary()` method returns a human-readable report.

In [ ]:
result = forecast([100, 120, 130, 115, 140, 160, 150, 170], steps=5)
print(result.summary())

## 5. Compare All Models

Every forecast evaluates 30+ models internally. See the full ranking.

In [ ]:
from vectrix import compare, loadSample

df = loadSample("airline")
ranking = compare(df, date="date", value="passengers", steps=12)
ranking

`compare()` returns a DataFrame sorted by MAPE with columns: `model`, `mape`, `rmse`, `mae`, `smape`, `time_ms`, `selected`.

## 6. Get All Model Forecasts

Sometimes you want predictions from every model, not just the winner.

In [ ]:
result = forecast(df, date="date", value="passengers", steps=12)

all_preds = result.allForecasts()
all_preds.head()

## 7. Export Results

Convert results to DataFrame, CSV, or JSON.

In [ ]:
df_result = result.toDataframe()
df_result

In [ ]:
json_str = result.toJson()
print(json_str[:200])

## 8. Complete Example

Full end-to-end workflow: forecast, inspect metrics, compare models, export.

In [ ]:
from vectrix import forecast

monthly_sales = [
    450, 470, 520, 540, 580, 620, 590, 610, 650, 680, 710, 750,
    460, 490, 530, 560, 600, 640, 610, 630, 670, 700, 730, 770,
]

result = forecast(monthly_sales, steps=6)

print(f"Model: {result.model}")
print(f"MAPE: {result.mape:.2f}%")
print(f"Next 6 months: {result.predictions.round(1)}")
print(f"Lower bound:   {result.lower.round(1)}")
print(f"Upper bound:   {result.upper.round(1)}")
print()
print("Top 5 models:")
print(result.compare().head(5))

## Result Object Reference

| Attribute / Method | Type | Description |
|---|---|---|
| `.predictions` | `np.ndarray` | Forecast values |
| `.dates` | `list` | Forecast date strings |
| `.lower` | `np.ndarray` | 95% lower confidence bound |
| `.upper` | `np.ndarray` | 95% upper confidence bound |
| `.model` | `str` | Selected model name |
| `.mape` | `float` | Validation MAPE (%) |
| `.rmse` | `float` | Validation RMSE |
| `.mae` | `float` | Validation MAE |
| `.smape` | `float` | Validation sMAPE |
| `.models` | `list` | All evaluated model names |
| `.compare()` | `DataFrame` | All models ranked by MAPE |
| `.allForecasts()` | `DataFrame` | Every model's predictions |
| `.summary()` | `str` | Formatted text summary |
| `.describe()` | `DataFrame` | Pandas-style statistics |
| `.toDataframe()` | `DataFrame` | date, prediction, lower95, upper95 |
| `.toJson(path)` | `str` | Export to JSON |

---

**Next:** [Tutorial 02 — Analysis & DNA](02_analyze.ipynb)